Bước 1: Cài đặt thư viện môi trường

In [ ]:
pip install selenium webdriver-manager requests beautifulsoup4 pandas langchain-text-splitters underthesea

Bước 2: Chạy đoạn mã hoàn chỉnh

In [ ]:
import os
import json
import re
import time
import random  # Thêm thư viện này để xếp hàng các luồng
import concurrent.futures
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote

# Thư viện Selenium (Đã xóa bỏ hoàn toàn webdriver_manager)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# Thư viện Langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# 1. CẤU HÌNH BAN ĐẦU & DANH MỤC TỈNH THÀNH
# ==========================================
PROVINCES_MAP = {
    "AG": "An Giang", "BR": "Bà Rịa - Vũng Tàu", "BG": "Bắc Giang", "BK": "Bắc Kạn", "BL": "Bạc Liêu",
    "BN": "Bắc Ninh", "BT": "Bến Tre", "BD": "Bình Định", "BP": "Bình Phước", "BU": "Bình Thuận",
    "CM": "Cà Mau", "CB": "Cao Bằng", "CT": "Cần Thơ", "DN": "Đà Nẵng", "DL": "Đắk Lắk",
    "DG": "Đắk Nông", "DB": "Điện Biên", "DP": "Đồng Nai", "DT": "Đồng Tháp", "GL": "Gia Lai",
    "HG": "Hà Giang", "HD": "Hải Dương", "HP": "Hải Phòng", "HN": "Hà Nội", "HT": "Hà Tĩnh",
    "HB": "Hòa Bình", "HY": "Hưng Yên", "KH": "Khánh Hòa", "KG": "Kiên Giang", "KT": "Kon Tum",
    "LC": "Lai Châu", "LS": "Lạng Sơn", "LO": "Lào Cai", "LD": "Lâm Đồng", "LA": "Long An",
    "ND": "Nam Định", "NA": "Nghệ An", "NB": "Ninh Bình", "NT": "Ninh Thuận", "PT": "Phú Thọ",
    "PY": "Phú Yên", "QB": "Quảng Bình", "QM": "Quảng Nam", "QN": "Quảng Ninh", "QG": "Quảng Ngãi",
    "QT": "Quảng Trị", "ST": "Sóc Trăng", "SL": "Sơn La", "TN": "Tây Ninh", "TB": "Thái Bình",
    "TY": "Thái Nguyên", "TH": "Thanh Hóa", "TTH": "Thừa Thiên Huế", "TG": "Tiền Giang", "TV": "Trà Vinh",
    "TQ": "Tuyên Quang", "VL": "Vĩnh Long", "VP": "Vĩnh Phúc", "YB": "Yên Bái", "HCM": "TP Hồ Chí Minh",
    "HGI": "Hậu Giang", "BDU": "Bình Dương"
}

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# ==========================================
# 2. CÁC HÀM HỖ TRỢ CÀO DỮ LIỆU CỐT LÕI
# ==========================================
def clean_vietnamese_text(text):
    if not text:
        return ""
    text = re.sub(r'(\s+)', ' ', text)
    text = re.sub(r'(Xem tiếp|Chia sẻ|Lượt xem:|Ngày đăng:|Tác giả:).*', '', text, flags=re.IGNORECASE)
    return text.strip()

def extract_article_details(driver, article_url):
    try:
        driver.get(article_url)
        time.sleep(1.5) 
        soup = BeautifulSoup(driver.page_source, "html.parser")

        title_node = soup.find(["h1", "h2"]) or soup.find(class_=re.compile(r"title|heading", re.I))
        title = title_node.get_text(strip=True) if title_node else "No Title"
        if "Thông tin Website" in title:
            title = "No Title"

        paragraphs = soup.find_all("p")
        text_content = "\n".join([p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 20])
        return title, clean_vietnamese_text(text_content)
    except Exception:
        return None, None

def crawl_province_data(province_name, driver, max_articles=100):
    articles_data = []
    page = 1
    encoded_province = quote(province_name)

    while len(articles_data) < max_articles:
        search_url = f"https://vietnamtourism.gov.vn/search?txtkey={encoded_province}&page={page}"
        try:
            driver.get(search_url)
            time.sleep(4) 
            soup = BeautifulSoup(driver.page_source, "html.parser")
            links = soup.find_all("a", href=True)
            
            article_links = []
            for l in links:
                href = l['href']
                if ("/post/" in href or re.search(r'\d{4,}', href)) and "search" not in href and "page=" not in href and "method=" not in href:
                    article_links.append(href)
            
            article_links = list(set(article_links))
            if not article_links:
                break

            for link in article_links:
                if len(articles_data) >= max_articles:
                    break
                full_url = link if link.startswith("http") else f"https://vietnamtourism.gov.vn{link}"
                title, content = extract_article_details(driver, full_url)
                
                if content and len(content) > 1600 and title != "No Title":
                    articles_data.append({"title": title, "text_content": content, "url": full_url})
            page += 1
        except Exception as e:
            print(f"  [ERROR] Lỗi tại trang {page}: {e}")
            break

    return articles_data[:max_articles]

# ==========================================
# 3. HÀM XỬ LÝ ĐỘC LẬP CHO TỪNG TỈNH
# ==========================================
def process_single_province(prefix, province_name):
    output_path = f"data/raw/{prefix}_raw.json"

    if os.path.exists(output_path) and os.path.getsize(output_path) > 10:
        print(f"[SKIP] Đã có dữ liệu cho {province_name} ({prefix}).")
        return f"Bỏ qua {province_name}"

    # LỆNH CHỐNG TREO: Bắt luồng nghỉ ngẫu nhiên 1-4 giây trước khi bật Chrome
    time.sleep(random.uniform(1, 4))
    print(f"[EXECUTION] Đang xử lý: {province_name} ({prefix})...")
    
    chrome_options = Options()
    chrome_options.add_argument("--headless=new") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2
    }
    chrome_options.add_experimental_option("prefs", prefs)

    # KHỞI TẠO TRỰC TIẾP, Không dùng Service hay ChromeDriverManager nữa
    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(45)

    try:
        raw_data = crawl_province_data(province_name, driver, max_articles=100)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(raw_data, f, ensure_ascii=False, indent=4)
        print(f"  -> [SUCCESS] Lưu thành công {len(raw_data)} bài cho {province_name}")
    except Exception as e:
        print(f"  -> [ERROR] Lỗi tại {province_name}: {e}")
    finally:
        driver.quit()
        
    return f"Hoàn thành {province_name}"

# ==========================================
# 4. LUỒNG THỰC THI CHÍNH (CÓ BATCHING & COOLDOWN)
# ==========================================
def main():
    print("\n[START] Bắt đầu quá trình thu thập dữ liệu đa luồng (Chế độ Chia mẻ an toàn)...")
    start_time = time.time()

    BATCH_SIZE = 4       
    COOLDOWN_TIME = 15   

    province_items = list(PROVINCES_MAP.items())

    for i in range(0, len(province_items), BATCH_SIZE):
        batch = province_items[i:i + BATCH_SIZE]
        print(f"\n==================================================")
        print(f"[BATCH] Đang khởi chạy mẻ {i // BATCH_SIZE + 1} (Gồm các tỉnh: {', '.join([p[0] for p in batch])})")
        print(f"==================================================")

        with concurrent.futures.ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
            futures = []
            for prefix, province_name in batch:
                # Đã trả lại về đúng 2 tham số như nguyên bản
                futures.append(executor.submit(process_single_province, prefix, province_name))
            
            for future in concurrent.futures.as_completed(futures):
                future.result() 

        if i + BATCH_SIZE < len(province_items):
            print(f"\n[COOLDOWN] Đã xong mẻ. Đang nghỉ {COOLDOWN_TIME} giây để hạ nhiệt máy chủ...")
            for sec in range(COOLDOWN_TIME, 0, -1):
                print(f"  -> Tiếp tục sau {sec} giây...", end="\r")
                time.sleep(1)
            print("  -> Bắt đầu mẻ mới!                  ")

    end_time = time.time()
    print(f"\n[COMPLETE] Thu thập dữ liệu thô hoàn tất sau {round(end_time - start_time, 2)} giây.")
    print("\n[START] Chuyển sang giai đoạn chuẩn hóa và Chunking...")

    CHUNK_SIZE = 600
    CHUNK_OVERLAP = 100
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len, is_separator_regex=False)
    processed_records = []

    for prefix, province_name in PROVINCES_MAP.items():
        raw_file_path = f"data/raw/{prefix}_raw.json"
        if not os.path.exists(raw_file_path): continue
        
        with open(raw_file_path, "r", encoding="utf-8") as f:
            try: articles = json.load(f)
            except Exception: continue

        for doc_idx, article in enumerate(articles):
            doc_id = f"{prefix}_{str(doc_idx + 1).zfill(3)}"
            title = article.get("title", "No Title")
            text_body = article.get("text_content", "")
            url = article.get("url", "")
            if not text_body.strip(): continue

            chunks = text_splitter.split_text(text_body)
            for chunk_idx, chunk_text in enumerate(chunks):
                chunk_id = f"{doc_id}_chunk_{chunk_idx + 1}"
                processed_records.append({
                    "doc_id": doc_id, "chunk_id": chunk_id, "province": province_name,
                    "title": title, "text_content": chunk_text, "url": url
                })

    if processed_records:
        final_dataset_df = pd.DataFrame(processed_records)
        csv_output_path = "data/processed/vietnam_tourism_rag_dataset.csv"
        final_dataset_df.to_csv(csv_output_path, index=False, encoding="utf-8-sig")
        print("\n" + "="*50)
        print("[SUCCESS] Toàn bộ Pipeline đã hoàn thành!")
        print(f"[METRICS] Tổng số chunk tạo ra: {len(final_dataset_df)}")
        print(f"[REPOSITORY] File lưu tại: {csv_output_path}")
        print("="*50)
    else:
        print("[WARNING] Không có dữ liệu nào được chunking.")

if __name__ == "__main__":
    main()


[INFO] Đang tải Chrome Driver chung (chỉ tải 1 lần duy nhất để chống treo máy)...

[START] Bắt đầu quá trình thu thập dữ liệu đa luồng (Chế độ Chia mẻ an toàn)...

[BATCH] Đang khởi chạy mẻ 1 (Gồm các tỉnh: AG, BR, BG, BK)
[EXECUTION] Đang xử lý: An Giang (AG)...
[EXECUTION] Đang xử lý: Bà Rịa - Vũng Tàu (BR)...
[EXECUTION] Đang xử lý: Bắc Giang (BG)...
[EXECUTION] Đang xử lý: Bắc Kạn (BK)...
